# TrackLink USBL — Resolved from Logs

Visualises resolved TrackLink USBL data from the log-based pipeline:
ship track, target positions, ship→target bearing arrows, and time-series
of depth, horizontal range, and inclination angle.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
DEPLOYMENT: str = "r7jjskxq_20131022_004934"

# qd61g27j_20110410_011202
# qd61g27j_20120422_043114
# qd61g27j_20130414_013620

# qdc5ghs3_20120501_033336
# qdc5ghs3_20130405_103429

# qdch0ftq_20110415_020103
# qdch0ftq_20120430_002423
# qdch0ftq_20130406_023610
# qdch0ftq_20140327_071251

# qdchdmy1_20110416_005411
# qdchdmy1_20130406_081713
# qdchdmy1_20140328_063358

# qtqxshxs_20110815_102540

# r234xgje_20120530_064545
# r234xgje_20140616_205232

# r23685bc_20120530_233021
# r23685bc_20140616_225022

# r23m7ms0_20120601_070118
# r23m7ms0_20140616_044549

# r29mrd12_20110612_045149
# r29mrd12_20130611_015335

# r29mrd5h_20110612_033752
# r29mrd5h_20130611_002419

# r7jjskxq_20121013_060425
# r7jjskxq_20131022_004934

# r7jjss8n_20121013_060425
# r7jjss8n_20131022_004934

# r7jjssbh_20121013_060425
# r7jjssbh_20131022_004934

RESOLVED_DIR: Path = Path("/data/exos_01/acfr_tracklink_logs_v3_resolved")
USBL_FILE_WITH: Path = RESOLVED_DIR / f"{DEPLOYMENT}_tracklink_fixes_with_extrinsics.csv"
USBL_FILE_WITHOUT: Path = RESOLVED_DIR / f"{DEPLOYMENT}_tracklink_fixes_without_extrinsics.csv"

## Load data

In [ ]:
REQUIRED_COLS: list[str] = [
    "timestamp",
    "ship_latitude",
    "ship_longitude",
    "ship_heading",
    "ship_roll",
    "ship_pitch",
    "target_bearing_angle",
    "target_slant_range",
    "target_depth",
    "target_height",
    "target_horizontal_range",
    "target_inclination_angle",
    "target_latitude",
    "target_longitude",
    "horizontal_position_std",
    "depth_position_std",
]


def _load(path: Path) -> pd.DataFrame:
    dataframe: pd.DataFrame = pd.read_csv(
        path, parse_dates=["timestamp"], date_format="ISO8601"
    )
    missing: list[str] = [col for col in REQUIRED_COLS if col not in dataframe.columns]
    if missing:
        raise ValueError(f"CSV is missing required columns: {missing}")
    return dataframe


usbl_with: pd.DataFrame = _load(USBL_FILE_WITH)
usbl_without: pd.DataFrame = _load(USBL_FILE_WITHOUT)

print(f"Deployment           : {DEPLOYMENT}")
print(f"Rows (with extr.)    : {len(usbl_with)}")
print(f"Rows (without extr.) : {len(usbl_without)}")
usbl_with.head(3)

In [ ]:
def _extrinsics_label(dataframe: pd.DataFrame) -> str:
    return (
        f"loc=({dataframe['usbl_extrinsics_locx'].iloc[0]:.3f}, "
        f"{dataframe['usbl_extrinsics_locy'].iloc[0]:.3f}, "
        f"{dataframe['usbl_extrinsics_locz'].iloc[0]:.3f}) m  "
        f"rot=({np.degrees(dataframe['usbl_extrinsics_rotx'].iloc[0]):.2f}, "
        f"{np.degrees(dataframe['usbl_extrinsics_roty'].iloc[0]):.2f}, "
        f"{np.degrees(dataframe['usbl_extrinsics_rotz'].iloc[0]):.2f}) deg"
    )


label_with: str = _extrinsics_label(usbl_with)
label_without: str = _extrinsics_label(usbl_without)
print("With extrinsics   :", label_with)
print("Without extrinsics:", label_without)

## Overview map: ship track and target positions

In [ ]:
t_s: np.ndarray = (usbl_with["timestamp"].astype(np.int64) / 1e9).to_numpy()
elapsed: np.ndarray = (t_s - t_s.min()) / 60.0

center_lat: float = float(usbl_with["target_latitude"].mean())
center_lon: float = float(usbl_with["target_longitude"].mean())

colorscale: str = "Viridis"

fig = go.Figure()

fig.add_trace(
    go.Scattermapbox(
        lat=usbl_with["ship_latitude"],
        lon=usbl_with["ship_longitude"],
        mode="lines+markers",
        marker=dict(
            size=4,
            color=elapsed,
            colorscale=colorscale,
            cmin=0,
            cmax=float(elapsed.max()),
            showscale=True,
            colorbar=dict(title="Elapsed (min)", x=0.92),
        ),
        line=dict(width=1, color="royalblue"),
        name="Ship track",
        hovertemplate=(
            "<b>Ship</b><br>"
            "Lat: %{lat:.6f}<br>"
            "Lon: %{lon:.6f}<br>"
            "<extra></extra>"
        ),
    )
)

fig.add_trace(
    go.Scattermapbox(
        lat=usbl_with["target_latitude"],
        lon=usbl_with["target_longitude"],
        mode="markers",
        marker=dict(
            size=6,
            color=elapsed,
            colorscale=colorscale,
            cmin=0,
            cmax=float(elapsed.max()),
            showscale=False,
        ),
        name="USBL target",
        customdata=np.stack(
            [
                usbl_with["target_depth"],
                usbl_with["target_horizontal_range"],
                usbl_with["horizontal_position_std"],
            ],
            axis=1,
        ),
        hovertemplate=(
            "<b>USBL target</b><br>"
            "Lat: %{lat:.6f}<br>"
            "Lon: %{lon:.6f}<br>"
            "Depth: %{customdata[0]:.1f} m<br>"
            "Horizontal range: %{customdata[1]:.1f} m<br>"
            "Horizontal σ: %{customdata[2]:.2f} m<br>"
            "<extra></extra>"
        ),
    )
)

fig.update_layout(
    title=f"{DEPLOYMENT} (TrackLink logs, with extrinsics)<br><sup>{label_with}</sup>",
    mapbox=dict(
        style="open-street-map",
        center=dict(lat=center_lat, lon=center_lon),
        zoom=14,
    ),
    legend=dict(x=0.01, y=0.99),
    height=700,
    margin=dict(l=0, r=0, t=60, b=0),
)

fig.show()

In [ ]:
t_s_wo: np.ndarray = (usbl_without["timestamp"].astype(np.int64) / 1e9).to_numpy()
elapsed_wo: np.ndarray = (t_s_wo - t_s_wo.min()) / 60.0

center_lat_wo: float = float(usbl_without["target_latitude"].mean())
center_lon_wo: float = float(usbl_without["target_longitude"].mean())

fig_wo = go.Figure()

fig_wo.add_trace(
    go.Scattermapbox(
        lat=usbl_without["ship_latitude"],
        lon=usbl_without["ship_longitude"],
        mode="lines+markers",
        marker=dict(
            size=4,
            color=elapsed_wo,
            colorscale=colorscale,
            cmin=0,
            cmax=float(elapsed_wo.max()),
            showscale=True,
            colorbar=dict(title="Elapsed (min)", x=0.92),
        ),
        line=dict(width=1, color="royalblue"),
        name="Ship track",
        hovertemplate=(
            "<b>Ship</b><br>"
            "Lat: %{lat:.6f}<br>"
            "Lon: %{lon:.6f}<br>"
            "<extra></extra>"
        ),
    )
)

fig_wo.add_trace(
    go.Scattermapbox(
        lat=usbl_without["target_latitude"],
        lon=usbl_without["target_longitude"],
        mode="markers",
        marker=dict(
            size=6,
            color=elapsed_wo,
            colorscale=colorscale,
            cmin=0,
            cmax=float(elapsed_wo.max()),
            showscale=False,
        ),
        name="USBL target",
        customdata=np.stack(
            [
                usbl_without["target_depth"],
                usbl_without["target_horizontal_range"],
                usbl_without["horizontal_position_std"],
            ],
            axis=1,
        ),
        hovertemplate=(
            "<b>USBL target</b><br>"
            "Lat: %{lat:.6f}<br>"
            "Lon: %{lon:.6f}<br>"
            "Depth: %{customdata[0]:.1f} m<br>"
            "Horizontal range: %{customdata[1]:.1f} m<br>"
            "Horizontal σ: %{customdata[2]:.2f} m<br>"
            "<extra></extra>"
        ),
    )
)

fig_wo.update_layout(
    title=f"{DEPLOYMENT} (TrackLink logs, without extrinsics)<br><sup>{label_without}</sup>",
    mapbox=dict(
        style="open-street-map",
        center=dict(lat=center_lat_wo, lon=center_lon_wo),
        zoom=14,
    ),
    legend=dict(x=0.01, y=0.99),
    height=700,
    margin=dict(l=0, r=0, t=60, b=0),
)

fig_wo.show()

## Time series: target XYZ in vessel frame — with vs without extrinsics

In [ ]:
fig_xyz = make_subplots(
    rows=3,
    cols=2,
    shared_xaxes=True,
    column_titles=(
        f"With extrinsics<br><sup>{label_with}</sup>",
        f"Without extrinsics<br><sup>{label_without}</sup>",
    ),
    row_titles=("X (m)", "Y (m)", "Z (m)"),
    vertical_spacing=0.06,
    horizontal_spacing=0.08,
)

vessel_axes: list[tuple[str, str, int]] = [
    ("target_x_vessel", "steelblue", 1),
    ("target_y_vessel", "darkorange", 2),
    ("target_z_vessel", "seagreen", 3),
]

for col, color, row in vessel_axes:
    for dataframe, col_idx in [(usbl_with, 1), (usbl_without, 2)]:
        fig_xyz.add_trace(
            go.Scatter(
                x=dataframe["timestamp"],
                y=dataframe[col],
                mode="lines+markers",
                marker=dict(size=3),
                line=dict(color=color),
                showlegend=False,
            ),
            row=row,
            col=col_idx,
        )

fig_xyz.update_layout(
    title=f"Target XYZ in vessel frame — {DEPLOYMENT} (TrackLink logs)",
    height=700,
)

fig_xyz.show()